# Cloud-edge experiment frontend
The sole editable configuration, including operational controls, is `config/experiment.json`. The current configuration enables preparation and submission of the approved full fresh rerun. Run All submits that graph once; subsequent submissions reuse its recorded identity. Scientific/source changes invalidate previous approvals; live guard changes do not. Resume requires explicit opt-in and submission permission.

## 1. Load and optionally prepare
Preparation creates the frozen plan only when `workflow.experiment.prepare_enabled` is explicitly enabled. Otherwise open an existing local experiment, or leave it unset.

In [ ]:
from pathlib import Path
from cloud_edge_rl.config import load_config
from cloud_edge_rl.experiment import Experiment

PROJECT_ROOT = Path.cwd()
CONFIG = load_config(root=PROJECT_ROOT)
controls = CONFIG['workflow']['experiment']
experiment_directory = PROJECT_ROOT / controls['directory']
experiment = None
RESULTS = {}
if controls['prepare_enabled']:
    experiment = Experiment.prepare(CONFIG, PROJECT_ROOT, directory=experiment_directory)
elif experiment_directory.is_dir():
    experiment = Experiment.open(experiment_directory, root=PROJECT_ROOT)
print({'config': 'config/experiment.json', 'directory': str(experiment_directory), 'experiment_available': experiment is not None})

## 2. Submit or explicitly resume
Submission permission is required for both operations. `resume_enabled` selects resume instead of submit; historical approvals do not authorize new execution. Scoped admission still applies.

In [ ]:
if controls['submission_enabled']:
    if experiment is None:
        raise RuntimeError('Prepare or open an experiment before explicit submission')
    if controls.get('resume_enabled', False):
        RESULTS['submission'] = experiment.resume()
    else:
        RESULTS['submission'] = experiment.submit()
else:
    print('Submission disabled in config/experiment.json')

## 3. Status
The current configuration enables cloud status refresh. While the graph is running, collection and verification report pending. After cloud completion, rerun cells 3–6 to download and verify; do not enable resume for an already completed graph.

In [ ]:
if experiment is not None:
    RESULTS['status'] = experiment.status(refresh=controls['refresh_enabled'])
else:
    RESULTS['status'] = {'state': 'not_prepared', 'directory': str(experiment_directory)}
print(RESULTS['status'])

## 4. Collect
With explicit JSON opt-in, collect only after remote status confirms call completion. Otherwise report pending; never retry compute.

In [ ]:
if controls['collection_enabled']:
    remote_status = RESULTS['status'].get('remote-status', {})
    if experiment is not None and remote_status.get('call_completed') is True:
        RESULTS['collection'] = experiment.collect()
        RESULTS['status'] = experiment.status(refresh=False)
    else:
        print('Collection pending: remote call completion not confirmed')
else:
    print('Collection disabled in config/experiment.json')

## 5. Verify
With explicit JSON opt-in, verify only a transport-complete artifact transfer bound to the current submission attempt and experiment. Otherwise report pending.

In [ ]:
if controls['verification_enabled']:
    local_status = RESULTS['status']
    transfer = local_status.get('transfer', {})
    submission = local_status.get('submission', {})
    if (experiment is not None and transfer.get('transport_complete') is True
            and submission.get('attempt_id') is not None
            and transfer.get('attempt_id') == submission['attempt_id']
            and transfer.get('experiment_id') == local_status.get('experiment_id')):
        RESULTS['verification'] = experiment.verify()
    else:
        print('Verification pending: current-attempt artifact transfer not complete')
else:
    print('Verification disabled in config/experiment.json')

## 6. Results
Display returned status and artifacts only; this cell performs no inference, submission, or implicit resume.

In [ ]:
print(RESULTS)